# Varian (2014): Econometrics × Machine Learning — Integrated Synthetic Banking Lab

**Classification:** Foundational / Methodological / Conceptual Review.  
**Source:** Hal R. Varian (2014), *Big Data: New Tricks for Econometrics*, JEP 28(2), 3–28. DOI: https://doi.org/10.1257/jep.28.2.3

This is a **teaching simulation** using invented banks and outcomes. No real supervisory data or observed distress events are included. The notebook consolidates exercises on prediction vs inference, data checks, nonlinear patterns, overfitting, CART, ensembles, penalization, variable stability, time-series evaluation, fairness/proxies, counterfactuals and model uncertainty. Results must not be represented as evidence about real banks.

## 0. Experimental protocol

1. Confirm synthetic provenance and data dictionary.
2. Separate predictive questions from descriptive, inferential and causal questions.
3. Engineer **only point-in-time features** and exclude contemporaneous or future outcomes.
4. Fix train (2018–2020), validation (2021), and test (2022) by *feature date*; impose six-month gaps between partitions to avoid overlapping target windows.
5. Fit transparent baselines, CART, Random Forest, Gradient Boosting and regularized regression.
6. Select models on validation only; inspect the test once after selection.
7. Evaluate rare-event discrimination, calibration, interpretability, stability and sensitivity.
8. Record assumptions and approvals required before any application to restricted data.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance
from sklearn.model_selection import ParameterGrid
SEED = 20261015
np.random.seed(SEED)
print('Libraries imported. All data below are synthetic.')

## 1. Data provenance, loading and descriptive profiling

The committed CSV contains 15 fictitious banks observed monthly over 60 months (2018–2022). `event` is a simulated monthly event and `distress_next_6m` indicates whether an event occurs in the next six months; labels in the final six months are missing because the future window is unobserved. **Never use `event` as a predictor of the future label in this exercise**; it is supplied to audit target construction.

In [ ]:
candidate_paths = [Path('data/synthetic_varian_bank_panel.csv'), Path('experiments/varian-econometrics-ml/data/synthetic_varian_bank_panel.csv'), Path('/mnt/data/varian_study/synthetic_varian_bank_panel.csv')]
path = next((p for p in candidate_paths if p.exists()), None)
if path is None:
    raise FileNotFoundError('Run from the notebook directory or repository root; the companion CSV is required.')
df = pd.read_csv(path, parse_dates=['date'])
assert df.bank_id.nunique() == 15 and len(df) == 900
assert df.groupby('bank_id').size().eq(60).all()
assert not df.duplicated(['bank_id','date']).any()
print(df.shape, 'bank-months; synthetic outcome prevalence:', df.distress_next_6m.mean().round(3))
display(df.head()); display(df.describe(include='all').T)

### Exercise A — Define the research purpose

For each question, decide whether it is **prediction, summarization, estimation, or hypothesis testing**: (a) Which banks face an event in six months? (b) What bank profiles emerge? (c) What is the association between capital and future event risk? (d) Is a specified coefficient distinguishable from zero? Explain why none alone establishes a causal effect.

## 2. Feature engineering and information leakage

Lagged changes and moving averages use only information available at the observation month. The forward-looking target is provided separately and must never enter the feature matrix. We deliberately include capital, NPL, liquidity, profitability and macro stress to demonstrate interpretable and nonlinear models.

In [ ]:
df = df.sort_values(['bank_id','date']).copy()
for col in ['car','npl','liquidity']:
    df[f'{col}_change_3m'] = df.groupby('bank_id')[col].diff(3)
    df[f'{col}_avg_3m'] = df.groupby('bank_id')[col].transform(lambda s: s.rolling(3, min_periods=3).mean())
df['car_gap_13'] = df.car - 13
df['npl_gap_10'] = df.npl - 10
df['capital_npl_interaction'] = df.car_gap_13 * df.npl_gap_10
features = ['car','npl','liquidity','roa','macro_stress','car_change_3m','npl_change_3m','liquidity_change_3m','car_avg_3m','npl_avg_3m','car_gap_13','npl_gap_10','capital_npl_interaction']
assert 'distress_next_6m' not in features and 'event' not in features
assert df.loc[df.month_index >= 54,'distress_next_6m'].isna().all()
print('Point-in-time predictors:', features)

## 3. Chronological splits with label-horizon embargo

Six-month-ahead targets require careful split boundaries. Train feature months 0–23; validation 30–35; test 42–53. Six-month gaps prevent the training target window from overlapping the validation feature window and the validation target window from overlapping the test feature window. These dates are illustrative and deliberately conservative.

In [ ]:
train = df[(df.month_index >= 3) & (df.month_index <= 23) & df.distress_next_6m.notna()].copy()
valid = df[(df.month_index >= 30) & (df.month_index <= 35) & df.distress_next_6m.notna()].copy()
test  = df[(df.month_index >= 42) & (df.month_index <= 53) & df.distress_next_6m.notna()].copy()
assert train.month_index.max()+6 < valid.month_index.min()
assert valid.month_index.max()+6 < test.month_index.min()
assert train.date.max() < valid.date.min() < test.date.min()
for name, part in [('train',train),('validation',valid),('test',test)]:
    print(name, len(part), 'events:', int(part.distress_next_6m.sum()), 'rate:', round(part.distress_next_6m.mean(),3))
Xtr,ytr = train[features],train.distress_next_6m.astype(int)
Xva,yva = valid[features],valid.distress_next_6m.astype(int)
Xte,yte = test[features],test.distress_next_6m.astype(int)

## 4. Baselines, CART, Random Forest and boosting

The logistic baseline is deliberately regularized; all models use training data only. The validation set is used for model comparison. The final test set is reserved for a single post-selection evaluation.

In [ ]:
models = {
 'logistic_L2': Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler()),('model',LogisticRegression(C=1, max_iter=2000, random_state=SEED))]),
 'tree_depth_3': Pipeline([('impute',SimpleImputer(strategy='median')),('model',DecisionTreeClassifier(max_depth=3,min_samples_leaf=20,random_state=SEED))]),
 'random_forest': Pipeline([('impute',SimpleImputer(strategy='median')),('model',RandomForestClassifier(n_estimators=150,max_depth=5,min_samples_leaf=8,random_state=SEED,n_jobs=-1))]),
 'gradient_boosting': Pipeline([('impute',SimpleImputer(strategy='median')),('model',GradientBoostingClassifier(n_estimators=80,learning_rate=.05,max_depth=2,random_state=SEED))])}
def score(y,p):
    return {'PR_AUC':average_precision_score(y,p),'ROC_AUC':roc_auc_score(y,p) if len(np.unique(y))==2 else np.nan,'Brier':brier_score_loss(y,p)}
validation_results=[]
for name,model in models.items():
    model.fit(Xtr,ytr)
    pred=model.predict_proba(Xva)[:,1]
    validation_results.append({'model':name,**score(yva,pred)})
validation_table=pd.DataFrame(validation_results).sort_values('PR_AUC',ascending=False)
display(validation_table)
chosen=validation_table.iloc[0]['model'];print('Chosen on validation PR-AUC:',chosen)

In [ ]:
selected=models[chosen]
test_prob=selected.predict_proba(Xte)[:,1]
print('One-time final held-out test evaluation:',score(yte,test_prob))
threshold=.5
ConfusionMatrixDisplay(confusion_matrix(yte,test_prob>=threshold)).plot()
plt.title('Synthetic test confusion matrix (threshold 0.50)');plt.show()
print('Recall:',round(recall_score(yte,test_prob>=threshold,zero_division=0),3),'Precision:',round(precision_score(yte,test_prob>=threshold,zero_division=0),3))

### Exercise B — Overfitting and tree pruning

Fit trees with maximum depths 1–8 on the training data. Plot training versus validation PR-AUC. Choose depth from **validation** only. Why is it incorrect to select depth based on the final test set?

In [ ]:
depth_rows=[]
for depth in range(1,9):
    model=Pipeline([('impute',SimpleImputer(strategy='median')),('tree',DecisionTreeClassifier(max_depth=depth,min_samples_leaf=8,random_state=SEED))])
    model.fit(Xtr,ytr)
    depth_rows.append({'depth':depth,'train_PR_AUC':average_precision_score(ytr,model.predict_proba(Xtr)[:,1]),'validation_PR_AUC':average_precision_score(yva,model.predict_proba(Xva)[:,1])})
depth_df=pd.DataFrame(depth_rows);display(depth_df)
depth_df.set_index('depth').plot(marker='o',title='Tree complexity and synthetic-data overfitting');plt.show()

## 5. Regularization and variable-selection stability

L1 penalization may set coefficients to zero; L2 shrinks coefficients. Selection can change across samples and should not be mistaken for causal importance.

In [ ]:
rows=[]
for penalty in ['l1','l2']:
 for C in [.03,.1,1,10]:
    pipe=Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler()),('model',LogisticRegression(penalty=penalty,solver='liblinear',C=C,max_iter=2000,random_state=SEED))])
    pipe.fit(Xtr,ytr)
    rows.append({'penalty':penalty,'C':C,'validation_PR_AUC':average_precision_score(yva,pipe.predict_proba(Xva)[:,1]),'nonzero_coefficients':int(np.sum(np.abs(pipe.named_steps['model'].coef_[0])>1e-6))})
display(pd.DataFrame(rows).sort_values('validation_PR_AUC',ascending=False))

In [ ]:
rng=np.random.default_rng(SEED)
selection=[]
for repeat in range(20):
    sampled_banks=rng.choice(train.bank_id.unique(),size=train.bank_id.nunique(),replace=True)
    boot=pd.concat([train[train.bank_id==bank] for bank in sampled_banks],ignore_index=True)
    if boot.distress_next_6m.nunique()<2:continue
    pipe=Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler()),('model',LogisticRegression(penalty='l1',solver='liblinear',C=.1,max_iter=2000,random_state=SEED))])
    pipe.fit(boot[features],boot.distress_next_6m.astype(int))
    selection.append((np.abs(pipe.named_steps['model'].coef_[0])>1e-6).astype(int))
if selection:
    display(pd.Series(np.mean(selection,axis=0),index=features,name='bootstrap_selection_frequency').sort_values(ascending=False))

## 6. Interpretation and permutation importance

Permutation importance measures change in predictive score when a predictor is shuffled. Correlated features can share information, reducing individual importance. It is not a causal-effect estimate.

In [ ]:
imp=permutation_importance(selected,Xva,yva,n_repeats=8,random_state=SEED,scoring='average_precision')
display(pd.Series(imp.importances_mean,index=features,name='validation_PR_AUC_drop').sort_values(ascending=False).head(10))

## 7. Summarization versus inference: descriptive profiles

Create four capital/NPL quadrants using illustrative thresholds (13% CAR, 10% NPL). These are teaching thresholds, not a regulatory classification or empirical finding. Contrast a descriptive grouping with a predictive score.

In [ ]:
def risk_quadrant(x):
    return ('capital_adequate' if x.car>=13 else 'capital_low')+' / '+('npl_low' if x.npl<10 else 'npl_high')
profile=test.copy();profile['risk_profile']=profile.apply(risk_quadrant,axis=1)
display(profile.groupby('risk_profile').agg(bank_months=('bank_id','size'),synthetic_event_rate=('distress_next_6m','mean')).sort_values('synthetic_event_rate',ascending=False))

## 8. Classification fairness and proxies — methodological discussion

The dataset contains no protected demographic attributes. Therefore **no demographic fairness or discrimination conclusion can be drawn**. As in Varian's HMDA example, low predictor importance does not demonstrate absence of discrimination. Before a real credit fairness study, define the protected attributes, legal standard, relevant decision stage, selection process, permissible covariates and approvals.

**Exercise C:** Explain why removing a protected attribute from a model does not necessarily remove proxy discrimination. Why might conditioning on some variables mask a discriminatory pathway?

## 9. Time-series and counterfactual exercises

Varian introduces Bayesian Structural Time Series for alternative-data forecasting and intervention counterfactuals. This notebook demonstrates a **descriptive pre/post comparison only**, not a causal effect estimate. For causal claims, specify a credible intervention, counterfactual and identification assumptions.

In [ ]:
monthly=df.groupby('month_index',as_index=False).agg(mean_npl=('npl','mean'),mean_car=('car','mean'),macro_stress=('macro_stress','mean'))
monthly.plot(x='month_index',y=['mean_npl','mean_car'],title='Synthetic sector time series');plt.show()
pre=monthly[monthly.month_index<34].mean_npl.mean();post=monthly[(monthly.month_index>=34)&(monthly.month_index<=43)].mean_npl.mean()
print('Descriptive post-minus-pre NPL difference (NOT causal):',round(post-pre,3))

## 10. Model uncertainty and specification sensitivity

Compare the held-out validation PR-AUC of several plausible feature sets. Do not cherry-pick a single favorable specification; document all tested choices. This is a predictive analogue of specification sensitivity, not a causal robustness analysis.

In [ ]:
feature_sets={'financial_core':['car','npl','liquidity','roa'],'plus_macro':['car','npl','liquidity','roa','macro_stress'],'full_engineered':features}
spec=[]
for name,cols in feature_sets.items():
    pipe=Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler()),('model',LogisticRegression(max_iter=2000,random_state=SEED))])
    pipe.fit(train[cols],ytr)
    spec.append({'specification':name,**score(yva,pipe.predict_proba(valid[cols])[:,1])})
display(pd.DataFrame(spec))

## 11. Consolidated knowledge check

1. Classify: prediction, summarization, estimation or hypothesis testing for four hypothetical supervisory questions.
2. Explain why training accuracy may improve while test accuracy deteriorates.
3. Explain how CART differs from logistic regression and why neither is necessarily superior.
4. Distinguish bagging, Random Forests and boosting.
5. Contrast ridge, LASSO, elastic net and Bayesian variable selection.
6. Why do future labels require an embargo when splitting panel time series?
7. Why does low importance of a demographic variable not prove absence of discrimination?
8. What assumptions are required before a BSTS counterfactual can be interpreted causally?
9. Explain sampling uncertainty versus model uncertainty.
10. List five controls required before working with approved supervisory data.

## 12. Real-data readiness gate — NOT EXECUTED

**Do not substitute confidential supervisory data into this public notebook.** Before any real-data analysis: (i) obtain authorization and ethical/institutional approvals; (ii) use an approved restricted computing environment; (iii) document data lineage, reporting vintages and event labels; (iv) check entity/time dependence and overlapping target windows; (v) implement rolling-origin and subgroup evaluation; (vi) agree minimum performance and calibration standards; (vii) assess explainability, model governance, security and confidentiality; (viii) independently review results. Keep public outputs synthetic or aggregated and approved.

## Reference

Varian, H. R. (2014). Big data: New tricks for econometrics. *Journal of Economic Perspectives, 28*(2), 3–28. https://doi.org/10.1257/jep.28.2.3

This notebook is a **research training artifact**, not a validated supervisory decision system.